# Aufgabe 1: Reinforcement Learning mit Policy Network

![CartPole](https://www.gymlibrary.dev/_images/cart_pole.gif)

[CartPole](https://www.gymlibrary.dev/environments/classic_control/cart_pole/) ist ein Kontrollproblem, bei dem es darum geht, ein auf einem Wagen montiertes Pendel zu balancieren.

Es eignet sich als einfaches Einstiegsbeispiel für Reinforcement Learning, da der Zustandsraum mit vier Dimensionen 

- Position des Wagens,
- Geschwindigkeit des Wagens,
- Winkel des Pendels, und
- Winkelgeschwindigkeit des Pendels

sehr klein und die Anzahl der Aktionen

- Wagen nach links schieben oder
- Wagen nach rechts schieben

überschaubar ist.

In der ersten Aufgabe geht es darum, das Pendel mit einem einfachen *Policy Network* auszubalancieren.

In [ ]:
import numpy as np
print(np.__version__)

Gym scheint mit NumPy 2.0 oder höher inkompatibel sein. Falls dies der Fall ist, kann ein Downgrade von NumPy helfen:

In [ ]:
!pip install numpy==1.26.4

In [ ]:
# show matplotlib version (if needed)
#!pip list | grep matplotlib

Zunächst installieren wir [`gymnasium`](https://gymnasium.farama.org/) (An universal API for reinforcement learning environments) , [`pygame`](https://www.pygame.org/docs/), `moviepy`, `pysdl2` und `pyvirtualdisplay`.

In [ ]:
!pip install gymnasium    # new version of gymnasium
!pip install pygame
!pip install moviepy
!pip install pysdl2
!pip install pyvirtualdisplay

In [ ]:
import random 
import torch

from torch import nn
import torch.nn.functional as F
from torch.distributions import Categorical

import matplotlib.pyplot as plt      # plotting

import gymnasium as gym              # new Gymnasium version
from gymnasium import wrappers

from IPython import display
from tqdm.notebook import tqdm       # progress bar

## Rendering im Jupyter Notebook

Normalerweise benötigt man für das Rendern eine "richtige" Anwendung. Wir behelfen uns hier mit `matplotlib`. Die Funktion [`env.render()`](https://gymnasium.farama.org/api/env/#gymnasium.Env.render) gibt ein Objekt zurück, das mit `matplotlib` dargestellt werden kann.

In [ ]:
def render(env, img, observation=None):
    img.set_data(env.render())
    display.display(plt.gcf())
    display.clear_output(wait=True)

def render_additional_output(env, img, observation=None):
    img.set_data(env.render())
    display.display(plt.gcf())
    display.clear_output(wait=True)
    print('Cart pos. ' + str(np.round(observation[0],2)) + 
          ', veloc. ' + str(np.round(observation[1],2)) +
          ', pole angle ' + str(np.round(np.degrees(observation[2]),2)) + ' degree' +
          ', pole angle veloc. ' + str(np.round(observation[2],2)))

Zunächst erzeugen wir ein *Environment*.

## Baseline: `RandomPolicy`

Die folgende Policy macht einfach zufällige Aktionen, die somit unabhängig von den Beobachtungen (`observation`) sind.

In [ ]:
class RandomPolicy:
    
    def __call__(self, observation):
        return random.choice([0, 1])
    
    def update(self, *args):
        # Do nothing
        pass # do nothing
    
    def init_game(self, observation):
        pass  # do nothing

**Observation Space**

Die Beobachtungen liegen in einem `ndarray` der Größe (4,) vor (Variable `observation`) mit den folgenden Wertebereichen für Position und Geschwindigkeit:

Num |  Observation          | Min                 | Max
:--:|-----------------------|:-------------------:|:----------------:
0   | Cart Position         | -4.8                | 4.8
1   | Cart Velocity         | -Inf                | Inf
2   | Pole Angle            | ~ -0.418 rad (-24°) | ~ 0.418 rad (24°)
3   | Pole Angular Velocity |-Inf                 | Inf

**Belohnung**

Eine Belohnung von $+1$ wird für jeden Step erziehlt in dem der Stab aufrecht steht, d.h. weniger als $\pm 12^\circ$ schief steht.

$475$ Belohnungen sind in v1 zu erreichen.

In [ ]:
def play_game(policy, episodes=2000, do_render = False, seed=100):
    random.seed(seed)
    torch.manual_seed(seed) # Sets the seed for generating random numbers on all devices
    if do_render:
        env = gym.make("CartPole-v1", render_mode="rgb_array") # create environment
    else:
        env = gym.make("CartPole-v1")                          # create environment
    observation, info = env.reset(seed=seed)
    policy.init_game(observation)

    if do_render:
        plt.ion()                       # switch interactive mode on
        plt.axis('off')                 # don't show axes
        img = plt.imshow(env.render())  # show "a frame"
   
    status = {}
    episode = 0
    status['steps'] = 0
    status['episode_reward'] = 0
    status['average_reward'] = 0
    total_reward = 0
    
    with tqdm(total=episodes) as pbar:
        pbar.set_postfix(status)
        while True:
            try:
                action = policy(observation)
                observation, reward, terminated, truncated, info = env.step(action)
                status['steps'] += 1
                status['episode_reward'] += reward
                if do_render:
                    #render_additional_output(env, img, observation) # additionally printing observation vector
                    render(env, img)
                policy.update(observation, reward, terminated, truncated, info, pbar)

                if terminated or status['steps'] > 1000: 
                    episode += 1
                    if episode > pbar.total:
                        break
                    total_reward += status['episode_reward']
                    status['average_reward'] = 0.05 * status['episode_reward'] + (1 - 0.05) * status['average_reward']
                    if status['average_reward'] > env.spec.reward_threshold:
                        print(f"Solved! Running reward is now {status['average_reward']} and "
                              f"the last episode runs to {status['steps']} time steps!")
                        break

                    pbar.set_postfix(status, refresh=episode % 10 == 0)
                    pbar.update()
                    status['steps'] = 0
                    
                    status['episode_reward'] = 0
                    observation, info = env.reset()
                    policy.init_game(observation)

            except KeyboardInterrupt:
                break
    env.close()

In [ ]:
policy = RandomPolicy()
play_game(policy, episodes=10, do_render=True)
#play_game(policy, episodes=20, do_render=True)

## Aufgabe 1.1: Policy Network

Für unser erstes Model benötigen wir ein Policy-Network, das den vierdiemesionalen Zustand in zwei Aktionen übersetzt. 
Das Modell so wie folgt aussehen:

1. Ein `Linear` Layer mit `hidden_size` als Zieldimension und `ReLU` als Aktivierungsfunktion,
2. Als `policy_head` ein `Linear` Layer mit `n_actions` (in diesem Fall $2$) als Zieldimension und `Softmax` als Aktivierungsfunktion.

In [ ]:
class PolicyNetwork(nn.Module):

    def __init__(self, hidden_size=128):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(4, hidden_size),  # inpit size = 4 (4 observations)
            nn.ReLU()
        )
        self.policy = nn.Sequential(
            nn.Linear(hidden_size, 2),
            nn.Softmax(dim=-1)
        )
        
    def forward(self, x):
        x = self.fc(x)
        x = self.policy(x)
        return x

### Strategie

Die Funktion `__call__(self, observation)` berechnet die Aktion wie folgt:

- Die Wahrscheinlichkeiten `probs` werden mit dem `PolicyNet` (hier als `self.net` erreichbar) berechnet,
- es wird mit [`torch.distrib.Categorical`](https://pytorch.org/docs/stable/distributions.html#categorical) eine passende Wahrscheinlichkeitsverteilung ([kategorische Verteilung](https://www.statology.org/categorical-distribution/), wie z.B. ein Würfel) erzeugt und mit `sample()` eine Aktion "ausgewürfelt",
- in `self.memory` wird der Logarithmus der Wahrscheinlichkeit (`m.log_prob(acttion)`) gespeichert (für das spätere Training).

### Update des Modells

Das Training findet jeweils am Ende einer Spielepisode statt:

- Zunächst werden die *diskontierten Belohnungen* berechnet,
- diese werden so skaliert, dass sie normalverteilt sind.
- Der Verlust der Policy ergibt sich als `- reward * log_prob`,
- mit der Summe der Verluste wird ein `optimizer.step()` durchgeführt.

In [ ]:
from collections import namedtuple
SavedAction = namedtuple('SavedAction', ['log_prob'])
    
class SimplePolicy:
    
    def __init__(self, gamma=0.99, lr=5e-3):
        # Two possible actions 0, 1
        self.ACTIONS = [0, 1]       
        self.net = PolicyNetwork()
        self.optimizer = torch.optim.Adam(self.net.parameters(), lr=lr)
        self.mean_reward = None
        self.games = 0
        self.gamma = gamma                         # reward discount
        self.eps = np.finfo(np.float32).eps.item() # smallest float32
        
    def __call__(self, observation):
 
        probs = self.net(torch.tensor(observation))
        m = Categorical(probs)
        action = m.sample()
        
        self.memory.append(SavedAction(m.log_prob(action)))
        
        return self.ACTIONS[action.item()]
        
    def init_game(self, observation):
        self.memory = []
        self.rewards = []
        self.total_reward = 0
        
    def discount_rewards(self, r):
        discounted = torch.zeros(len(r))
        summe = 0
        for t in reversed(range(0, len(r))):
            summe = summe * self.gamma + r[t]
            discounted[t] = summe
        return discounted
        
    def update(self, observation, reward, terminated, truncated, info, status):
        self.total_reward += reward
        self.rewards.append(reward)
        if terminated:
            self.games += 1
            if self.mean_reward is None:
                self.mean_reward = self.total_reward
            else:
                self.mean_reward = self.mean_reward * 0.95 + self.total_reward * (1.0 - 0.95)
            
            self.optimizer.zero_grad()
                
            # calculate discounted reward and make it normal distributed
            discounted = []
            R = 0
            for r in self.rewards[::-1]:
                R = r + self.gamma * R
                discounted.insert(0, R)
            discounted = torch.tensor(discounted)
            discounted = (discounted - discounted.mean()) / (discounted.std() + self.eps)
            
            policy_losses = []
            for mem, discounted_reward in zip(self.memory, discounted):
                policy_losses.append(-(mem.log_prob * discounted_reward))
                
            loss = torch.stack(policy_losses).sum()
            loss.backward()    
            self.optimizer.step()
            
            if self.games % 1000 == 0:
                self.save(f"model_{self.games}.pt")
    
    
    def load(self, PATH):
        checkpoint = torch.load(PATH)
        self.net.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.games = checkpoint['games']
        self.mean_reward = checkpoint['mean_reward']
        
    def save(self, PATH):
        torch.save({
                    'games': self.games,
                    'model_state_dict': self.net.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'mean_reward': self.mean_reward}, PATH)

In [ ]:
policy = SimplePolicy()
play_game(policy)

In [ ]:
play_game(policy, episodes=10, do_render=True)

## Aufgabe 1.1: Actor-Critic-Modell

Für das Actor-Critic-Modell erhält unser Netzwerk einen weiteren Ausgang `critic`, der eine Schätzung des Werts eines Zustands liefern soll.

In [ ]:
class ActorCriticNetwork(nn.Module):

    def __init__(self, hidden_size=32):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(4, hidden_size),
            nn.ReLU()
        )
        self.policy = nn.Sequential(
            nn.Linear(hidden_size, 2),
            nn.Softmax(dim=-1)
        )
        self.critic = nn.Sequential(
            nn.Linear(hidden_size, 1)
        )
        
        
    def forward(self, x):
        x = self.fc(x)
        p = self.policy(x)
        v = self.critic(x)
        return p, v

### Strategie

Die Strategie berechnet nun auch den Wert des Zustands mithilfe des Netzwerks und speichert diesen für das Training.

### Training

Die Verlustfunktion besteht nun aus zwei Teilen:

1. Der `policy_loss` summiert `log_prob * advantage`, wobei `advantage = discounted_reward - value` ist. 
   Diese Differenz wird auch als *temporal difference* bezeichnet.
2. Der `value_loss` summiert die Abweichung zwischen `value` und diskontiertem Reward. 
   Dabei wird meistens `F.smooth_l1_loss` verwendet.

In [ ]:
from collections import namedtuple
SavedAction = namedtuple('SavedAction', ['log_prob', 'value'])
    
class ACPolicy:
    
    def __init__(self, gamma=0.99, lr=5e-3):
        # Two possible actions 0, 1
        self.ACTIONS = [0, 1]       
        self.net = ActorCriticNetwork()
        self.optimizer = torch.optim.Adam(self.net.parameters(), lr=lr)
        self.mean_reward = None
        self.games = 0
        self.gamma = gamma
        self.eps = np.finfo(np.float32).eps.item()

        
    def __call__(self, observation):
 
        probs, value = self.net(torch.tensor(observation))
        m = Categorical(probs)
        action = m.sample()
        
        self.memory.append(SavedAction(m.log_prob(action), value))
        self.last_observation = observation
        
        return self.ACTIONS[action.item()]
        
    def init_game(self, observation):
        self.memory = []
        self.rewards = []
        self.total_reward = 0
        
        
    def update(self, observation, reward, terminated, truncated, info, status):
        self.total_reward += reward
        self.rewards.append(reward)
        if terminated:
            self.games += 1
            if self.mean_reward is None:
                self.mean_reward = self.total_reward
            else:
                self.mean_reward = self.mean_reward * 0.95 + self.total_reward * (1.0 - 0.95)
                
            # calculate discounted reward and make it normal distributed
            discounted = []
            R = 0
            for r in self.rewards[::-1]:
                R = r + self.gamma * R
                discounted.insert(0, R)
            discounted = torch.tensor(discounted)
            #discounted = (discounted - discounted.mean()) / (discounted.std() + self.eps)
            
            policy_losses = []
            value_losses = []
            for mem, discounted_reward in zip(self.memory, discounted):
                advantage = discounted_reward - mem.value.item() 
                policy_losses.append(-(mem.log_prob * advantage))
                
                value_losses.append(F.smooth_l1_loss(mem.value, discounted_reward.unsqueeze(0)))
               
            self.optimizer.zero_grad()
            loss = torch.stack(policy_losses).sum() + torch.stack(value_losses).sum()
            loss.backward()    
            self.optimizer.step()
            
            if self.games % 1000 == 0:
                self.save(f"model_{self.games}.pt")
    
    
    def load(self, PATH):
        checkpoint = torch.load(PATH)
        self.net.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.games = checkpoint['games']
        self.mean_reward = checkpoint['mean_reward']
        
    def save(self, PATH):
        torch.save({
                    'games': self.games,
                    'model_state_dict': self.net.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'mean_reward': self.mean_reward}, PATH)

In [ ]:
policy = ACPolicy()
play_game(policy)

In [ ]:
play_game(policy, do_render = True)